In [ ]:
import gymnasium as gym

from stable_baselines3 import SAC, A2C, DQN
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.utils import set_random_seed

In [ ]:

def make_env(env_id: str, rank: int, seed: int = 0, gravity: float = None):
    """
    Utility function for multiprocessed env.

    :param env_id: the environment ID
    :param num_env: the number of environments you wish to have in subprocesses
    :param seed: the initial seed for RNG
    :param rank: index of the subprocess
    """
    def _init():

        if gravity is not None:        
            env = gym.make(env_id, render_mode="human", g = gravity)
        else:
            env = gym.make(env_id, render_mode="human")
        env.reset(seed=seed + rank)
        return env
    set_random_seed(seed)
    return _init



In [ ]:
import random
if __name__ == "__main__":
    env_id = "CartPole-v1"
    # Number of processes to use
    num_env = 2
    # Create the vectorized environment

    #Here we have different gravities in each environment
    #This is where we can model heterogeneitity.
    gravityList = [10,10,10,.1]
    gravityList = [None, None, None, None]
    seed  = random.randint(0, 100)
    if num_env > 1:
        vec_env = SubprocVecEnv([make_env(env_id, i, seed, gravity = gravityList[i]) for i in range(num_env)])
        '''
        envList = []
        for i in range(num_env):
            env = gym.make(env_id, render_mode="rgb_array")
            seed = 123 + i
            env.reset(seed=seed, options={"low": -0.1, "high": 0.1}) 
            set_random_seed(seed)
            envList.append(env)
        vec_env = SubprocVecEnv(envList)
        '''
    else:
        i=0
        vec_env =make_env(env_id, i, gravity = gravityList[i])
    print("envelopes created")

    #model = PPO("MlpPolicy", vec_env, verbose=1)
    #model = SAC("MlpPolicy", vec_env, verbose=1)
    #model = A2C("MlpPolicy", vec_env, verbose=1)
    model = DQN("MlpPolicy", vec_env, verbose=1)
    print("model activated")
    
    model.learn(total_timesteps=10000)
    print("model learned")

    obs = vec_env.reset()
    for _ in range(1000):
        action, _states = model.predict(obs)
        obs, rewards, dones, info = vec_env.step(action)
        vec_env.render()
